# Transaction Foundation Model on Ray — Part 4: Pretrain with Ray Train

<div align="left">
  <a target="_blank" href="https://console.anyscale.com/template-preview/fintech_transaction_fm"><img src="https://img.shields.io/badge/🚀 Run_on-Anyscale-9hf"></a>&nbsp;
  <a href="https://github.com/anyscale/templates/tree/main/templates/fintech_transaction_fm" role="button"><img src="https://img.shields.io/static/v1?label=&message=View%20On%20GitHub&color=586069&logo=github&labelColor=2f363d"></a>&nbsp;
</div>

**⏱️ Time to complete**: ~5 min (full: ~2 h on 8×A10G)

---

Previously in Part 3, we turned the training data into tokens. Now we pretrain the foundation model: a transformer that reads the token stream and learns to predict the next token, the same way an LLM learns to predict the next word. The transactions themselves are the training signal — fraud labels play no part in this step.

We start by loading the three files Part 3 wrote, then Ray Train runs the training loop — one CPU worker at `mini`, eight GPUs at `full` — and we finish by saving the trained model to shared storage.

In [3]:
import sys, os, json

DEMO_ROOT = os.path.abspath(os.getcwd())
if DEMO_ROOT not in sys.path:
    sys.path.insert(0, DEMO_ROOT)

import logging
import ray
ray.init(ignore_reinit_error=True, logging_level=logging.ERROR,
         runtime_env={"working_dir": DEMO_ROOT,
                      # torch's native JIT wants a C compiler the workers don't have
                      "env_vars": {"TORCH_DISABLE_NATIVE_JIT": "1"}})

from src.paths import artifact_paths, get_demo_base_dir
from src.scale_config import load_scale

SCALE = "mini"                       # same knob as the earlier parts; full = 8×A10G, ~2h
cfg = load_scale(SCALE)
BASE = get_demo_base_dir()
paths = artifact_paths(BASE, SCALE)

## Load the training data

Part 3 wrote three files, and this cell loads them for training.

- `ids.npy` holds the tokens as one big integer array. Each row is one window from Part 3, a stretch of one card's history.
- `attn.npy` is the attention mask: it tells the model which positions to pay attention to (1) and which to ignore (0). A card's last window is usually not full, and the empty positions are the ones marked 0.
- `vocab.json` is the token list. The trainer reads it to size the model.

We wrap the two arrays into one Ray dataset. Ray Train splits it across the training workers.

In [4]:
import numpy as np

vocab_path = os.path.join(paths["nvcorpus"], "vocab.json")
ids = np.load(os.path.join(paths["nvcorpus"], "ids.npy"))
attn = np.load(os.path.join(paths["nvcorpus"], "attn.npy"))
n_seqs = int(ids.shape[0])

# Wrap the two arrays as one Ray dataset with the column names the training
# function expects. from_numpy makes a dataset out of an array; zip joins the
# two datasets column-wise.
train_ds = (ray.data.from_numpy(ids).rename_columns({"data": "input_ids"})
            .zip(ray.data.from_numpy(attn).rename_columns({"data": "attention_mask"})))

print(f"{n_seqs:,} windows × {ids.shape[1]} tokens each")
print(f"window 0, first 13 tokens as the model sees them: {ids[0][:13].tolist()}")
print("(the same window Part 3 decoded into AMT_3 MERCH_667 CAT_RETAIL ...)")

(ndarray_to_block pid=2927, ip=10.0.25.47) /tmp/ray/session_2026-07-23_14-00-03_908489_3132/runtime_resources/pip/7bda41e4671c1f0c7eee392064887c47d3f9e183/virtualenv/lib/python3.12/site-packages/cudf/utils/gpu_utils.py:162: UserWarning: No NVIDIA GPU detected
(ndarray_to_block pid=2927, ip=10.0.25.47)   warnings.warn("No NVIDIA GPU detected")


8 windows × 256 tokens each
window 0, first 13 tokens as the model sees them: [1, 8, 679, 2022, 2080, 2142, 2166, 2176, 2179, 2191, 3110, 3198, 3251]
(the same window Part 3 decoded into AMT_3 MERCH_667 CAT_RETAIL ...)


## What we're training

The model is a transformer from the Llama family — the same architecture behind most open LLMs — at NVIDIA's exact configuration: 29M parameters at `full`, a 2-layer miniature at `mini`. The details are in `src/model.py`.

Training is next-token prediction. At every position, the model reads the tokens so far and predicts the one that comes next; the difference between its guess and the real token is the training signal. This is what makes the model *causal*: every prediction uses only the past. Every position in a window is one prediction, so a single 4096-token window is ~4,000 training examples. Part 5 will use the trained model's internal state as the embedding; this notebook only trains it.

## Train with Ray Train

We train with an ordinary PyTorch loop: read a batch, forward pass, backward pass, optimizer step. The incidental pieces — building the model, the optimizer, the loss — are small helpers in `src/pretrain.py`. The loop itself is right here.

The loop is adapted for Ray in three places, numbered in the code below: (1) `prepare_model` wraps the PyTorch model for distributed training, (2) the batches come streaming from `get_dataset_shard` instead of a DataLoader, and (3) the loop is handed to `TorchTrainer` with a worker count and whether each worker gets a GPU. Ray distributes the rest — it starts the workers, with the cluster autoscaler adding nodes to fit them, averages their gradients after every step, and writes checkpoints to shared storage so an interrupted run can pick up from the last one.

`ScalingConfig` sets that worker count and the hardware: one CPU worker at `mini`, eight GPU workers at `full`. The training function does not change. Eight workers train on eight slices at once, so an epoch takes roughly an eighth of the time.

In [5]:
import torch
from src.pretrain import (build_pretrain_model, wrap_fsdp, make_optimizer,
                          make_lr_scheduler, next_token_loss, epoch_summary,
                          build_epoch_checkpoint)

def train_func(config: dict):
    """Runs on every training worker. Plain PyTorch plus the numbered Ray calls."""
    model = build_pretrain_model(config)
    wants_fsdp = config.get("use_fsdp", False) and torch.cuda.is_available()    # off at every preset
    model = wrap_fsdp(model) if wants_fsdp else ray.train.torch.prepare_model(model)   # (1)

    optimizer = make_optimizer(model, config)            # AdamW, NVIDIA's recipe
    scheduler = make_lr_scheduler(optimizer, config)     # warmup + cosine decay

    train_shard = ray.train.get_dataset_shard("train")   # (2) this worker's slice of the data
    for epoch in range(config["epochs"]):
        model.train()
        running, n_batches = 0.0, 0
        for batch in train_shard.iter_torch_batches(batch_size=config["batch_size"], dtypes=torch.long):
            loss = next_token_loss(model, batch)         # forward pass: next-token loss
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            if scheduler is not None:
                scheduler.step()
            running += float(loss.item())
            n_batches += 1

        metrics = epoch_summary(epoch, running / max(n_batches, 1), optimizer, config)
        ray.train.report(metrics, checkpoint=build_epoch_checkpoint(model, epoch, config))

In [6]:
from ray.train import ScalingConfig, RunConfig
from ray.train.torch import TorchTrainer
from src.pretrain import save_checkpoint
import math

pt = cfg["pretrain"]
seq_len = cfg["tokenize"]["seq_len"]

# Compute the total optimizer steps for the learning-rate schedule.
steps_per_epoch = max(1, math.ceil(n_seqs / pt["num_workers"] / pt["batch_size"]))
total_steps = steps_per_epoch * pt["epochs"]
warmup_steps = int(pt.get("warmup_ratio", 0.0) * total_steps)
print(f"{n_seqs:,} windows · global batch {pt['batch_size'] * pt['num_workers']} "
      f"· ~{steps_per_epoch} steps/epoch × {pt['epochs']} epochs = ~{total_steps:,} optimizer steps "
      f"(warmup {warmup_steps})")

8 windows · global batch 8 · ~1 steps/epoch × 2 epochs = ~2 optimizer steps (warmup 0)


In [7]:
trainer = TorchTrainer(
    train_func,                                       # (3) the loop above, run on every worker
    train_loop_config={
        "vocab_path": vocab_path, "arch": cfg["model"], "size": SCALE, "max_len": seq_len,
        "epochs": pt["epochs"], "batch_size": pt["batch_size"], "lr": pt["lr"],
        "use_fsdp": pt["use_fsdp"], "seed": 0,
        # Optimizer and warmup/cosine schedule settings — NVIDIA's recipe, from the scale config.
        "weight_decay": pt.get("weight_decay", 0.0), "betas": tuple(pt.get("betas", (0.9, 0.999))),
        "lr_schedule": pt.get("lr_schedule"), "total_steps": total_steps,
        "warmup_steps": warmup_steps, "min_lr_ratio": pt.get("min_lr_ratio", 0.0),
    },
    # (3) continued — worker count, and one GPU per worker or not:
    scaling_config=ScalingConfig(num_workers=pt["num_workers"], use_gpu=pt["use_gpu"]),
    datasets={"train": train_ds},                     # each worker gets a shard of this dataset
    run_config=RunConfig(
        name=f"transaction_fm_pretrain_{SCALE}",      # one run name per scale, so scales never resume each other
        storage_path=os.path.join(BASE, "ray_results"),  # shared storage: workers run on other nodes
    ),
)
result = trainer.fit()                                 # blocks until training finishes
save_checkpoint(result, paths["checkpoint"])           # copy the weights to the path Part 5 reads
print(f"done — final lm_loss {result.metrics['lm_loss']:.3f}, "
      f"perplexity {result.metrics['perplexity']:.1f} -> {paths['checkpoint']}")

(TrainController pid=7622) A run snapshot was found in storage folder at: '/mnt/cluster_storage/transaction-fm/ray_results/transaction_fm_pretrain_mini'
(TrainController pid=7622) This snapshot contains a list of checkpoints reported via `ray.train.report` and will be loaded. This allows the latest checkpoint found in the snapshot to be accessible within your training function via `ray.train.get_checkpoint`.
(TrainController pid=7622) If you meant to start a brand new training job without any information about previous checkpoints found in this directory, please configure a new, unique `RunConfig(name)` or delete the existing folder at '/mnt/cluster_storage/transaction-fm/ray_results/transaction_fm_pretrain_mini'.
(TrainController pid=7622) Requesting resources: {'CPU': 1} * 1
(TrainController pid=7622) [State Transition] INITIALIZING -> SCHEDULING.
(TrainController pid=7622) Attempting to start training worker group of size 1 with the following resources: [{'CPU': 1}] * 1
(RayTrainWor

(pid=8031) Running Dataset train_5_0.: 0.00 row [00:00, ? row/s]

(pid=8031) - Project:   0%|          | 0.00/1.00 [00:00<?, ? row/s]

(pid=8031) - Project:   0%|          | 0.00/1.00 [00:00<?, ? row/s]

(pid=8031) - ZipOperator(Project, Project):   0%|          | 0.00/1.00 [00:00<?, ? row/s]

...

(pid=8031) - Project:   0%|          | 0.00/1.00 [00:00<?, ? row/s]

(pid=8031) - ZipOperator(Project, Project):   0%|          | 0.00/1.00 [00:00<?, ? row/s]

(pid=8031) - split(1, equal=True):   0%|          | 0.00/1.00 [00:00<?, ? row/s]

(SplitCoordinator pid=8031) Registered dataset logger for dataset train_5_1
(SplitCoordinator pid=8031) Starting execution of Dataset train_5_1. Full logs are in /tmp/ray/session_2026-07-23_14-00-03_908489_3132/logs/ray-data
(SplitCoordinator pid=8031) Execution plan of Dataset train_5_1: InputDataBuffer[Input] -> TaskPoolMapOperator[Project], InputDataBuffer[Input] -> TaskPoolMapOperator[Project] -> ZipOperator[ZipOperator(Project, Project)] -> OutputSplitter[split(1, equal=True)]
(SplitCoordinator pid=8031) ✔️  Dataset train_5_1 execution finished in 0.05 seconds
(SplitCoordinator pid=8031) INFO:openlineage.client.client:OpenLineageClient will use `composite` transport
(SplitCoordinator pid=8031) INFO:openlineage.client.transport.composite:Stopping OpenLineage CompositeTransport emission after the first successful delivery because `continue_on_success=False`. Transport that emitted the event: <HttpTransport(name=first, kind=http, priority=1)>
(RayTrainWorker pid=3142, ip=10.0.25.47) 

done — final lm_loss 8.668, perplexity 5816.5 -> /mnt/cluster_storage/transaction-fm/model/mini/


## Read the results

Two numbers summarize the run: the training loss and its exponent, **perplexity** — the effective number of tokens the model is choosing between at each position. Guessing uniformly over the 6,251-token vocabulary would sit at perplexity 6,251; the `full` pretrain ends near **1.7**, fewer than two effective choices per token. That low a number is less surprising than it looks: several of the twelve fields (customer id, card index, state) repeat on nearly every transaction, so the model learns them immediately, and the remaining uncertainty concentrates in the informative fields — amount, merchant, time.

At `mini` the output above shows what two optimizer steps buy: the run starts almost exactly at the ceiling (a random init lands near it, even slightly above) and barely moves. That is all `mini` is for. The number that matters is the downstream fraud comparison in Part 6, built on the `full` pretrain.

In [8]:
m = result.metrics
print(f"final causal-LM loss {m['lm_loss']:.3f}   ·   perplexity {m['perplexity']:.1f}")

final causal-LM loss 8.668   ·   perplexity 5816.5


## Save the model for the later parts

Parts 5 through 8 load this model with standard Hugging Face tooling (NVIDIA's embedding code calls `AutoModelForCausalLM.from_pretrained`), so we save the trained weights in that format on shared storage.

In [9]:
import torch
from src.model import build_model

# Rebuild the model from the checkpoint, then save its inner Hugging Face model.
ck = paths["checkpoint"]
with open(os.path.join(ck, "model_config.json")) as f:
    mc = json.load(f)
m = build_model(vocab_path=os.path.join(ck, "vocab.json"), arch=mc["arch"], max_len=mc["max_len"])
sd = torch.load(os.path.join(ck, "model.pt"), map_location="cpu")
sd = sd["model_state_dict"] if isinstance(sd, dict) and "model_state_dict" in sd else sd
missing, unexpected = m.load_state_dict(sd, strict=False)
os.makedirs(paths["hf"], exist_ok=True)
m.lm.save_pretrained(paths["hf"])
print(f"exported HF decoder -> {paths['hf']}  (state_dict: missing {len(missing)}, unexpected {len(unexpected)})")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

exported HF decoder -> /mnt/cluster_storage/transaction-fm/model_hf/mini/  (state_dict: missing 0, unexpected 0)


(autoscaler +8m46s) [autoscaler] Downscaling node i-0ccc154c37275525f (node IP: 10.0.25.47) due to node idle termination.


## Scaling factors

Training cost is arithmetic: tokens × epochs × model size sets the work, and workers divide the wall-clock. At `full` that is 64,335 windows × 4096 tokens × 8 epochs through the 29M-parameter model — ~16,000 optimizer steps, about two hours on 8×A10G. Each worker trains on its own shard and the gradients average after every step, so eight workers run an epoch in roughly an eighth of the time.

Memory is the constraint that mattered in practice. The loss needs a prediction for every position — a (batch × 4096 × 6,251) tensor — and at batch 8 per worker that tensor alone overflowed the A10G's 15.6 GiB. `full` runs batch 4 with gradient checkpointing, which keeps the peak near 9 GiB, and the fix lives in the scale config, not the training loop. If the model itself ever outgrows one GPU, `use_fsdp` splits it across workers — at 29M parameters it fits easily.

The GPU workers exist for the two hours of this stage and scale back to zero afterward.

## Takeaways

We trained the foundation model: next-token prediction over the token windows, checkpointed to shared storage, and saved in Hugging Face format for the later parts. The training function is plain PyTorch; Ray Train supplied the workers, the data sharding, the gradient averaging, and the checkpointing, and `ScalingConfig` is the only difference between one CPU worker at `mini` and eight GPUs at `full`.

This is also the one stage NVIDIA's repo skips: their notebook trains a 30-step demonstration and downloads the real weights. Our ~16,000-step run is the real training — and the Part 1 results table shows what it is worth.

---

## Next

**Part 5 — Extract embeddings**: run every transaction through the trained model to get one embedding per transaction — CPU workers feed batches to GPU workers.